### DATA INGESTION

In [0]:
# Leitura da tabela.
df = spark.read.csv("/Volumes/workspace/default/retail-demand-intelligence/train.csv", header = True, inferSchema = True)

# Exibir os dados.
display(df)

# Exibir o esquema da tabela
df.printSchema()

In [0]:
# Contagem de linhas.
df.count()

In [0]:
# Contagem de colunas.
len(df.columns)

In [0]:
from pyspark.sql.functions import col, sum, when

# Valores nulos.
df_nulos = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(df_nulos)

In [0]:
from pyspark.sql.functions import min, max

# Intervalo de datas.

df_intervalo = df.select(
    min("Date").alias("data_minima"),
    max("Date").alias("data_maxima")
)

display(df_intervalo)

In [0]:
from pyspark.sql.functions import count

# Contagem de dados duplicados

df_duplicates = df.groupBy(df.columns).agg(count("*").alias("contagem")).filter(col("contagem") > 1)

df_duplicates.show()

### DATA PROFILING

In [0]:
# Quantas lojas existem?
store_number = df.select("Store").distinct().count()
store_number

In [0]:
# Verificar valores de cada variável categórica
print("Valores de Open:")
df.select("Open").distinct().show()
print("Valores de Promo:")
df.select("Promo").distinct().show()
print("Valores de StateHoliday:")
df.select("StateHoliday").distinct().show()
print("Valores de SchoolHoliday:")
df.select("SchoolHoliday").distinct().show()
print("Valores de DayOfWeek:")
df.select("DayOfWeek").distinct().show()


In [0]:
# Estatística para Sales
df.select("Sales").summary("mean", "stddev","min", "25%", "50%", "75%", "max").show()

# Estatística para Custormers
df.select("Customers").summary("mean", "stddev","min", "25%", "50%", "75%", "max").show()


In [0]:
# Contagem de Sales igual a 0.
df.filter(df.Sales == 0).count()

In [0]:
# Contagem de Open igual a 0.
df.filter(df.Open == 0).count()

Podemos afirmar que há 54 registros de diferença entre Sales = 0 e Open = 0. Logo existem dias que não venderam com a loja aberta


In [0]:
# Quantidade dias com a loja aberta sem vendas
df.filter(
    (df.Sales == 0) & (df.Open == 1)
).count()

In [0]:
# Quantidade de vendas com a loja fechada
df.filter(
    (df.Sales > 0) & (df.Open == 0)
).count()

Precisamos verificar se 0 vendas representa um comportamento válido do neegócio ou um problema de qualidade dos dados

In [0]:
#Verificar dados com Sales = 0 e Open = 1
df.filter(
    (df.Sales == 0) & (df.Open == 1)
).select(
    "Store",
    "DayOfWeek",
    "Date",
    "Sales",
    "Customers",
    "Promo",
    "StateHoliday",
    "SchoolHoliday"
).orderBy("Date").show(truncate=False) # Truncate permite que dados longos sejam exibidos na tebala ao invés de somente "...".

In [0]:
df.filter(
    (df.Sales == 0) & (df.Open == 1)
).groupby("Store").count().orderBy("count", ascending=False).show()

Parece ser um comportamento comum e não algo específico de uma loja. Logo devemos manter os 54 registros.


In [0]:
# Contagem de ocorrência de feriados
df.groupBy("StateHoliday").count().orderBy("StateHoliday").show()

In [0]:
# Contagem de ocorrência de lojas abertas/ fechadas por feriados.
df.groupBy("Open", "StateHoliday").count().orderBy("Open", "StateHoliday").show()

In [0]:
from pyspark.sql.functions import date_format

# Verificar se o dia da semana está correto.
df.select(
    # Converter data em dia da semana
    date_format(df.Date, "EEEE").alias("dia_da_semanda"),
    "DayOfWeek"
).distinct().orderBy("DayOfWeek").show()

In [0]:
# Quantos registros existem por loja
df.groupBy("Store").count().orderBy("count").show()

In [0]:
# Verificando se todas as lojas possuem registros do inicio ao fim do intervalo de datas
df.groupBy("Store").agg(
    min("Date").alias("data_inicio"),
    max("Date").alias("data_fim")
).orderBy("data_inicio").show()

In [0]:
# Valdiar a quantidade de registros por loja
df.groupBy("Store").count().groupBy("count").count().orderBy("count").show()

In [0]:
# Valdiar a data por loja
df.groupBy("Store").agg(
    min("Date").alias("data_inicio"),
    max("Date").alias("data_fim")
).groupBy(
    "data_inicio",
    "data_fim"
).count().show()

### Faltam dados para algumas lojas. Por isso, é necessário investigar se a ausência de dados está relacionada a abertura de lojas ou se pode ser um problema no dataset.

In [0]:
# Verificar todas as lojas que possuem 758 registros.
lojas_758 = (
    df.groupBy("Store").count().filter(col("count") == 758).orderBy("Store")
)

lojas_758.show()

In [0]:
# Verificar a contagem de dias abertos e fechados
df.filter(col("Store").isin(
    [row["Store"] for row in lojas_758.collect()])
).groupBy("Open").count().show()

In [0]:
# Verificar as datas de aberturas de cada loja
df.filter(col("Store").isin(
    [row["Store"] for row in lojas_758.collect()])
).groupBy("Store").agg(
    min("Date").alias("data_abertura"),
    max("Date").alias("data_fim")
).orderBy("data_abertura").show(200)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff

lojas_758 = (
    df.groupBy("Store").count().filter(col("count") == 758).select("Store")
)

# Agrupar os dados por loja sem reduzir a uma única linha de resultado
window_store = Window.partitionBy("Store").orderBy("Date")

# Identificar os gaps entre as datas
df_gaps = (
    # Realizar inner join para manter os registros com somente 758.
    df.join(lojas_758, on="Store", how="inner")
    # Lag acessa uma linha anterior ao resultaado atual. (No caso, um dia antes)
    .withColumn("data_anterior", lag(col("Date")).over(window_store))
    # Calcula o intervalo entre a data atual e a última data em que há registro
    .withColumn("dias_desde_anterior", datediff(col("Date"), col("data_anterior")))
)

In [0]:
# Contagem de ocorrência de gaps maiores que 1 dia.
df_gaps.filter(col("dias_desde_anterior") > 1).count()

In [0]:
# Visualizar quais lojas possuem esses gaps e qual o tamanho deles.
df_gaps.filter(
    col("dias_desde_anterior") > 1
).select(
    "Store",
    "Date",
    "data_anterior",
    "dias_desde_anterior"
).orderBy("Store", "Date").show(180)

In [0]:
# Verificar a contagem de datas com a ausência.
df_gaps.filter(
    col("dias_desde_anterior") > 1
).groupBy("data_anterior", "Date", "dias_desde_anterior").count().orderBy("dias_desde_anterior", ascending=False).show()

In [0]:
# Verificar se há somente um registro por dia em cada loja.
df.groupBy("Store", "Date").count().filter(col("count") > 1).count()

#EDA


In [0]:
from pyspark.sql.functions import (
    year,
    month,
    avg,
    sum,
    min,
    max
)

# Vendas por mês
vendas_mensais = (
    df.filter(col("Open") == 1)
    .groupBy(
        year("Date").alias("ano"),
        month("Date").alias("mes")
    )
    .agg(
        avg("Sales").alias("media_vendas"),
        sum("Sales").alias("total_vendas"),
        min("Sales").alias("min_vendas"),
        max("Sales").alias("max_vendas"),
    )
    .orderBy("ano", "mes")
)

vendas_mensais.show(50)

O mês de dezembro se apresenta como um período de vendas elevadas

In [0]:
# Vendas por dia da semana.
vendas_dia_semana = (
    df.filter(col("Open") == 1)
    .groupBy("DayOfWeek")
    .agg(
        avg("Sales").alias("media_vendas"),
        sum("Sales").alias("total_vendas"),
        min("Sales").alias("min_vendas"),
        max("Sales").alias("max_vendas"),
    )
    .orderBy("DayOfWeek")
)

vendas_dia_semana.show()

In [0]:
# Quantidade de dados por dia da semana
df.filter(col("Open") == 1).groupBy("DayOfWeek").count().orderBy("DayOfWeek").show()

Domingo apresenta uma média de vendas elevadas, porém com uma quantidade de vendas menor.


In [0]:
from pyspark.sql.functions import countDistinct

# Lojas abertas por dia na semana
df.filter(col("Open") == 1).groupBy("DayOfWeek").agg(countDistinct("Store")).alias("loajs_abertas").orderBy("DayOfWeek").show()

A baixa quantidade de vendas no domingo está relacionado a quantidade pequena de lojas abertas nesse período. Mesmo com um número menor de lojas abertas, o domingo apresenta excelentes indicadores.

In [0]:
# Verificando a influência das promoções no resultado final.
vendas_promo = (df.filter(col("Open") == 1)
                .groupBy("Promo")
                .agg(
                    avg("Sales").alias("media_vendas"),
                    sum("Sales").alias("total_vendas"),
                    count("*").alias("quantidade_vendas")
                )
                .orderBy("Promo")
                )

vendas_promo.show()

In [0]:
listCorr = ["Sales", "Customers", "Open", "Promo", "SchoolHoliday", "DayOfWeek"]

for i in listCorr:
    for j in listCorr:
        if i != j:
            corr = df.stat.corr(i, j)
            print(f"Correlação entre {i} e {j}: {corr}")

In [0]:
# Verificar média de vendas com ou sem promoção ao longo da semana
df.filter(col("Open") == 1).groupBy("DayOfWeek", "Promo").agg(
    avg("Sales").alias("media_vendas"),
    count("*").alias("qtde_lojas")
).orderBy("DayOfWeek").show()

In [0]:
# Verificar indicadoresd de vendas por loja.
df.filter(col("Open") == 1).groupBy("Store").agg(
    avg("Sales").alias("media_vendas"),
    sum("Sales").alias("total_vendas"),
    min("Sales").alias("min_vendas"),
    max("Sales").alias("max_vendas"),
    count("Sales").alias("qtde_vendas")
).orderBy("total_vendas").show(200)

In [0]:
vendas_loja = (
    df.filter(col("Open") == 1)
    .groupBy("Store")
    .agg(
        avg("Sales").alias("media_vendas"),
        count("*").alias("quantidade_observadas")
    )
    .orderBy(col("media_vendas").desc())
)

vendas_loja.show(180)

In [0]:
from pyspark.sql.functions import median, stddev

df.filter(col("Open") == 1).groupBy("Store").agg(
    avg("Sales").alias("media_vendas"),
    min("Sales").alias("min_vendas"),
    max("Sales").alias("max_vendas"),
    median("Sales").alias("mediana_vendas"),
    stddev("Sales").alias("desvio_vendas")
).orderBy("max_vendas", ascending=False).show(180)

In [0]:
vendas_loja.agg(
    avg("media_vendas").alias("media"),
    median("media_vendas").alias("mediana"),
    stddev("media_vendas").alias("desvio_padrao"),
    min("media_vendas").alias("min"),
    max("media_vendas").alias("max")
).show()

Percebemos que a maior média de um loja pode ser até 8 vezes que a média geral. Também é observado que a média geral é maior que a mediana, indicando que pode haver dados grandes que influenciam a média. 

In [0]:
# Top 10 lojas com maior média em vendas
vendas_loja.select(
    "Store",
    "media_vendas",
    ).orderBy(
        col("media_vendas").desc()
    ).show(10)

In [0]:
# Bottom 10 lojas com maior média em vendas.
vendas_loja.select(
    "Store",
    "media_vendas"
    ).orderBy(
        col("media_vendas").asc()
    ).show(10)

Store, Date, Promo são features relevantes para explicar a demanda. Custormers também apresenta forte associação, mas pode ser uma informação indisponível no momento da previsão.

# Feature Engineering

In [0]:
from pyspark.sql.functions import dayofmonth, weekofyear

df_features = (
    df
    .withColumn("ano", year(col("Date")))
    .withColumn("mes", month(col("Date")))
    .withColumn("dia", dayofmonth(col("Date")))
    .withColumn("semana", weekofyear(col("Date")))
)

df_features.select(
    "Date",
    "ano",
    "mes",
    "dia",
    "semana"
).show(10)

In [0]:
# Aplicando feature histórica de lag_1 (dia anterior).

df_features = (df_features.withColumn(
    "Sales_lag_1",
    lag(col("Sales")).over(Window.partitionBy("Store").orderBy("Date"))
).show(10))